# Algebra Module Showcase

A short tour of `autstr.algebra`: finite Boolean algebras and finite abelian groups as uniformly automatic classes.

This notebook shows three things:

1. how to build the algebra classes,
2. how to check concrete formulas on specific structures, and
3. how one compiled automaton can decide a property across an entire family.

In [1]:
from autstr.algebra import FiniteBooleanAlgebras, FiniteAbelianGroups

ba = FiniteBooleanAlgebras()
ab = FiniteAbelianGroups()

## Boolean algebras

The Boolean algebra with `n` atoms is represented by bitvectors of length `n`. The advice string is simply `1` repeated `n` times.

In [2]:
n = 4
print('advice:', ba.advice(n))
print('Meet({0,1}, {1,2}) = {1}:', ba.check('Meet(x,y,z)', n, x={0, 1}, y={1, 2}, z={1}))
print('Atom({3}) =', ba.check('Atom(x)', n, x={3}))

proper = 'exists x.(exists y.(Compl(x,y) and (not Leq(x,y))))'
dfa, tapes = ba.evaluate(proper)
print('uniform tapes:', tapes)
for size in [0, 1, 5]:
    advice = ba.advice(size)
    accepted = dfa.accepts([(s,) for s in advice])
    print(f'n={size}:', accepted)

advice: ['1', '1', '1', '1']
Meet({0,1}, {1,2}) = {1}: True
Atom({3}) = True
uniform tapes: ['advice']
n=0: False
n=1: True
n=5: True


## Finite abelian groups

A group like `Z_2 ⊕ Z_3` is encoded by LSB-first binary blocks, one block per cyclic factor, separated by `#`. The relation `A(x,y,z)` means `z = x + y`.

In [3]:
orders = [2, 3]
print('advice:', ab.advice(orders))
print('A((1,1),(1,2),(0,0)) =', ab.check('A(x,y,z)', orders, x=(1, 1), y=(1, 2), z=(0, 0)))
print('A((1,1),(1,2),(1,0)) =', ab.check('A(x,y,z)', orders, x=(1, 1), y=(1, 2), z=(1, 0)))

phi = ('exists x.((not A(x,x,x)) and '
       '(exists z.(A(x,x,z) and A(z,z,z))))')
dfa, tapes = ab.evaluate(phi)
print('uniform tapes:', tapes)
for orders in ([4], [3], [2, 3], [5, 6]):
    advice = ab.advice(orders)
    accepted = dfa.accepts([(s,) for s in advice])
    print(f'orders={orders}:', accepted)

advice: ['0', '1', '#', '1', '1', '#']
A((1,1),(1,2),(0,0)) = True
A((1,1),(1,2),(1,0)) = False
uniform tapes: ['advice']
orders=[4]: True
orders=[3]: False
orders=[2, 3]: True
orders=[5, 6]: True


## The localization $\mathbb{Z}[1/p]$ — one infinite group per prime

`z1p_localization(p)` is a factory: for every prime $p$ it produces the additive group
$\mathbb{Z}[1/p] = \{\, a/p^k : a \in \mathbb{Z},\ k \ge 0 \,\}$ together with an
**automatic presentation** of $(\mathbb{Z}[1/p], +)$.

The encoding writes $x = \pm(A + C)$ with integer part $A$ and fractional part
$C \in [0,1)$: a sign symbol followed by *pair digits* — position $i$ carries the digit of
$A$ at $p^{i-1}$ **and** the digit of $C$ at $p^{-i}$, so both expansions grow rightward
from the radix point and always align. Magnitude addition is regular because the
fractional overflow is guessed at the radix point and doubles as the carry into the
units digit; full signed addition `A(x,y,z)` is then *defined* by a first-order case
analysis over the signs, and `Z` (zero) and `Eq` are defined from `A` — the same
bootstrap the library uses for Büchi arithmetic. Signature: `A`, `N0` (non-negative),
`Z`, `Eq`.

In [4]:
from autstr.algebra import z1p_localization

z2 = z1p_localization(2)
%time presentation = z2.presentation  # built on first use
print('relations:', presentation.get_relation_symbols())
print('addition automaton:', presentation.automata['A'].num_states, 'states')

CPU times: user 248 ms, sys: 12 ms, total: 260 ms
Wall time: 260 ms
relations: ['U', 'N0', 'Z', 'A', 'Eq']
addition automaton: 73 states


### The string representation

Zero is the bare sign; `5/4 = 1 + 1/4` puts integer digit 1 and fraction digits `01`
into the pairs `(1,0)(0,1)`.

In [5]:
elements = [z2.element(0), z2.element(1), z2.element(-3),
            z2.from_fraction(5, 4), z2.from_fraction(-1, 2), z2.from_fraction(13, 8)]
for x in elements:
    print(f'{x.num}/2^{x.exp}'.rjust(7), '->', z2.encode(x))

  0/2^0 -> ['+']
  1/2^0 -> ['+', '1_0']
 -3/2^0 -> ['-', '1_0', '1_0']
  5/2^2 -> ['+', '1_0', '0_1']
 -1/2^1 -> ['-', '0_1']
 13/2^3 -> ['+', '1_1', '0_0', '0_1']


### Addition, checked against exact arithmetic

`check` assigns concrete elements to free variables and runs the convolution through
the query automaton. The exact-arithmetic layer (`add`, `neg`, ...) provides the
expected results.

In [6]:
half = z2.from_fraction(1, 2)
three_q = z2.from_fraction(3, 4)

print('1/2 + 3/4 = 5/4:  ', z2.check('A(x,y,z)', x=half, y=three_q, z=z2.add(half, three_q)))
print('-1/2 + 3/4 = 1/4: ', z2.check('A(x,y,z)', x=z2.neg(half), y=three_q, z=z2.from_fraction(1, 4)))
print('1/2 + 3/4 = 2 ?:  ', z2.check('A(x,y,z)', x=half, y=three_q, z=2))
print('Eq(2/4, 1/2):     ', z2.check('Eq(x,y)', x=(2, 2), y=half))
print('N0(-3):           ', z2.check('N0(x)', x=-3))

1/2 + 3/4 = 5/4:   True
-1/2 + 3/4 = 1/4:  True
1/2 + 3/4 = 2 ?:   False
Eq(2/4, 1/2):      True
N0(-3):            False


### The first-order theory distinguishes the primes

Every element of $\mathbb{Z}[1/p]$ is divisible by $p$ (halving just shifts the digit
pairs), but **not** by any other prime — e.g. $1/3 \notin \mathbb{Z}[1/2]$. Both facts
are first-order sentences, so the presentations decide them:

In [7]:
z3 = z1p_localization(3)

div2 = 'all x.(exists y.(A(y,y,x)))'                                # 2 | x for all x
div3 = 'all x.(exists y.(exists w.(A(y,y,w) and A(w,y,x))))'        # 3 | x for all x

for name, structure in [('Z[1/2]', z2), ('Z[1/3]', z3)]:
    print(f'{name}:  2-divisible: {structure.check(div2)}   '
          f'3-divisible: {structure.check(div3)}')

Z[1/2]:  2-divisible: True   3-divisible: False


Z[1/3]:  2-divisible: False   3-divisible: True


---

**Where to go from here.** The graph side of the library is showcased in
`treedepth_graphs.ipynb`: bounded tree-depth and bounded pathwidth graphs as uniformly
automatic classes, with full MSO over vertex sets — including a 6-state automaton that
decides bipartiteness for every graph of tree-depth ≤ 3. All of these are instances of
`autstr.uniform.UniformlyAutomaticClass`, which turns any advice-indexed family of
automata into a class with relativized query evaluation, sentence checking, and
member-structure instantiation.